# Kickstarter Campaign Analytics

**Module:** Big Data Management and Data Visualisation (STW7082CEM)  
**Student ID:** 250289  
**Dataset:** Kickstarter Projects (`ks-projects-201801.csv`) - 378,661 records x 15 columns  
**Tooling:** PySpark (exclusive), Tableau (visualisation)

## Objectives
1. **Classification** - predict campaign success (Logistic Regression baseline + Random Forest advanced)
2. **Regression** - predict pledged amount (Linear Regression baseline + GBT advanced)
3. **Clustering** - segment campaigns (K-Means)

## Reproducibility
- Random seed `42` is used for every stochastic operation (splits, models, cross-validation).
- Row counts are printed after each transformation to create an auditable data-flow trail.
- No data leakage: `pledged`/`state` are targets and are never used as predictive features.

---
# Phase 1: Data Loading and Validation

**Goal:** Ingest the raw CSV into a Spark DataFrame and confirm its integrity before any cleaning.

## 1.1 Imports and Random Seeds

All PySpark ML components are imported up front so later phases need no extra import cells. Python `random` and NumPy seeds are fixed to `42` for reproducibility (Spark operations are seeded individually).

In [1]:
# Core imports
import sys
import os
import random
import numpy as np
import pandas as pd

# PySpark core
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, isnan, lit,
    datediff, to_timestamp, month, dayofweek,
    log1p, expm1, sum as F_sum
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType,
    StringType, TimestampType, LongType
)

# PySpark ML (used in later phases)
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator,
    RegressionEvaluator, ClusteringEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Reproducibility: fix Python-side seeds (Spark ops seeded per-call)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Ensure PySpark uses the same Python interpreter as the notebook kernel
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(f"Python version : {sys.version.split()[0]}")
print(f"PySpark version: {pyspark.__version__}")
print(f"Random seed    : {SEED}")
print("All libraries imported successfully")

Python version : 3.12.10
PySpark version: 3.5.4
Random seed    : 42
All libraries imported successfully


## 1.2 SparkSession Initialisation

A local SparkSession is created using all available cores (`local[*]`). Driver memory and shuffle partitions are tuned for a single-machine workload, and adaptive query execution is enabled. The log level is set to `WARN` to keep the notebook output readable.

In [2]:
# Spark session -> local[*] uses all cores; memory/partitions tuned for single machine
spark = (
    SparkSession.builder
    .appName("KickstarterAnalytics")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

# Reduce log verbosity
spark.sparkContext.setLogLevel("WARN")

print("=" * 55)
print("SPARK ENVIRONMENT INITIALISED")
print("=" * 55)
print(f"PySpark Version : {spark.version}")
print(f"App Name        : {spark.sparkContext.appName}")
print(f"Master          : {spark.sparkContext.master}")
print(f"Parallelism     : {spark.sparkContext.defaultParallelism} cores")
print(f"Spark UI        : http://localhost:4040")
print("=" * 55)

SPARK ENVIRONMENT INITIALISED
PySpark Version : 3.5.4
App Name        : KickstarterAnalytics
Master          : local[*]
Parallelism     : 16 cores
Spark UI        : http://localhost:4040


## 1.3 Load the Dataset

An **explicit schema** is supplied rather than relying on `inferSchema`. A subset of rows in this dataset contain quoted/escaped text fields that cause Spark's inference to mis-type **every numeric column as a string** (which would break quantile/scaling operations later). The explicit schema fixes the numeric types deterministically, while `multiLine` + `escape` parse the quoted fields correctly so no rows are lost.

`deadline`/`launched` are read as strings here and converted to timestamps in Phase 2.3. 

In [3]:
# Data loading
input_path = "d:/Softwarica/Big Data/BigDataAssignment/Dataset/ks-projects-201801.csv"

# Explicit schema -> correct numeric types (inferSchema mis-types quoted fields); dates parsed in Phase 2.3
schema = StructType([
    StructField("ID", LongType(), True),
    StructField("name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("main_category", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("deadline", StringType(), True),
    StructField("goal", DoubleType(), True),
    StructField("launched", StringType(), True),
    StructField("pledged", DoubleType(), True),
    StructField("state", StringType(), True),
    StructField("backers", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("usd pledged", DoubleType(), True),       # note: space in column name (dataset quirk)
    StructField("usd_pledged_real", DoubleType(), True),
    StructField("usd_goal_real", DoubleType(), True),
])

try:
    df = (
        spark.read
        .option("header", "true")        # first row holds column names
        .option("multiLine", "true")     # allow quoted fields that span lines
        .option("escape", "\"")          # handle embedded double quotes
        .schema(schema)                  # explicit schema -> correct numeric types
        .csv(input_path)
    )
    # Cache: the raw DataFrame is reused across several validation steps
    df.cache()
    n_rows = df.count()   # triggers the read + populates the cache
    n_cols = len(df.columns)
    print(f"Data loaded successfully from:\n  {input_path}")
    print(f"Rows    : {n_rows:,}")
    print(f"Columns : {n_cols}")
except Exception as e:
    print(f"ERROR loading data from {input_path}")
    print(f"Reason: {e}")
    raise

Data loaded successfully from:
  d:/Softwarica/Big Data/BigDataAssignment/Dataset/ks-projects-201801.csv
Rows    : 378,661
Columns : 15


## 1.4 Validate Shape (Row & Column Counts)

Assertions guarantee the loaded data matches the expected dimensions. If the file is ever swapped or corrupted, the notebook fails loudly at this checkpoint rather than producing wrong downstream results.

In [ ]:
# Shape validation as claimed in datasaet
EXPECTED_ROWS = 378661
EXPECTED_COLS = 15

assert n_rows == EXPECTED_ROWS, f"Row count mismatch: got {n_rows:,}, expected {EXPECTED_ROWS:,}"
assert n_cols == EXPECTED_COLS, f"Column count mismatch: got {n_cols}, expected {EXPECTED_COLS}"

print(f"Row count verified    : {n_rows:,} == {EXPECTED_ROWS:,}")
print(f"Column count verified : {n_cols} == {EXPECTED_COLS}")
print("\nColumn names:")
for i, c in enumerate(df.columns, start=1):
    print(f"  {i:2d}. {c}")

Row count verified    : 378,661 == 378,661
Column count verified : 15 == 15

Column names:
   1. ID
   2. name
   3. category
   4. main_category
   5. currency
   6. deadline
   7. goal
   8. launched
   9. pledged
  10. state
  11. backers
  12. country
  13. usd pledged
  14. usd_pledged_real
  15. usd_goal_real


## 1.5 Inspect Schema and Sample Rows

`printSchema()` confirms the inferred data types, and `show(5)` gives a visual sanity check of the first records.

In [5]:
# Schema
print("=== Schema (explicit) ===")
df.printSchema()

print("=== First 5 Rows ===")
df.show(5, truncate=True)

=== Schema (explicit) ===
root
 |-- ID: long (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- deadline: string (nullable = true)
 |-- goal: double (nullable = true)
 |-- launched: string (nullable = true)
 |-- pledged: double (nullable = true)
 |-- state: string (nullable = true)
 |-- backers: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- usd pledged: double (nullable = true)
 |-- usd_pledged_real: double (nullable = true)
 |-- usd_goal_real: double (nullable = true)

=== First 5 Rows ===


+----------+--------------------+--------------+-------------+--------+----------+-------+-------------------+-------+--------+-------+-------+-----------+----------------+-------------+
|        ID|                name|      category|main_category|currency|  deadline|   goal|           launched|pledged|   state|backers|country|usd pledged|usd_pledged_real|usd_goal_real|
+----------+--------------------+--------------+-------------+--------+----------+-------+-------------------+-------+--------+-------+-------+-----------+----------------+-------------+
|1000002330|The Songs of Adel...|        Poetry|   Publishing|     GBP|2015-10-09| 1000.0|2015-08-11 12:12:28|    0.0|  failed|      0|     GB|        0.0|             0.0|      1533.95|
|1000003930|Greeting From Ear...|Narrative Film| Film & Video|     USD|2017-11-01|30000.0|2017-09-02 04:43:57| 2421.0|  failed|     15|     US|      100.0|          2421.0|      30000.0|
|1000004038|      Where is Hank?|Narrative Film| Film & Video|   

## 1.6 Null Value Distribution

Counting nulls per column early identifies which columns need cleaning in Phase 2 and confirms that the critical modelling columns (`goal`, `pledged`, `state`, `backers`) are usable. The check uses a single aggregation pass instead of looping the DataFrame per column.

In [6]:
# Null distribution -> count nulls for every column in one pass
null_counts_row = df.select([
    F_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
]).collect()[0]   # single-row result -> safe to collect

print("=== Null Counts per Column ===")
print(f"{'Column':<20} {'Nulls':>10} {'% of rows':>12}")
print("-" * 44)
for c in df.columns:
    nulls = null_counts_row[c]
    pct = (nulls / n_rows) * 100
    print(f"{c:<20} {nulls:>10,} {pct:>11.3f}%")

=== Null Counts per Column ===
Column                    Nulls    % of rows
--------------------------------------------
ID                            0       0.000%
name                          4       0.001%
category                      0       0.000%
main_category                 0       0.000%
currency                      0       0.000%
deadline                      0       0.000%
goal                          0       0.000%
launched                      0       0.000%
pledged                       0       0.000%
state                         0       0.000%
backers                       0       0.000%
country                       0       0.000%
usd pledged               3,797       1.003%
usd_pledged_real              0       0.000%
usd_goal_real                 0       0.000%


## 1.7 Summary Statistics and Duplicate ID Check

Descriptive statistics on the key numerical columns reveal scale and skew (important for later scaling and the log-transform of `pledged`). A duplicate-ID check confirms each row is a unique campaign.

In [7]:
# Summary statistics
print("=== Summary Statistics (key numerical columns) ===")
df.describe(["goal", "pledged", "backers", "usd_pledged_real", "usd_goal_real"]).show()

# State distribution (preview of the classification target)
print("=== Campaign State Distribution ===")
df.groupBy("state").count().orderBy(col("count").desc()).show()

# Duplicate ID check
distinct_ids = df.select("ID").distinct().count()
duplicates = n_rows - distinct_ids
print(f"Distinct IDs : {distinct_ids:,}")
print(f"Duplicate IDs: {duplicates:,}")
if duplicates == 0:
    print("No duplicate campaign IDs - each row is a unique campaign.")
else:
    print("WARNING: duplicate IDs present - investigate in Phase 2.")

=== Summary Statistics (key numerical columns) ===


+-------+-----------------+-----------------+------------------+-----------------+------------------+
|summary|             goal|          pledged|           backers| usd_pledged_real|     usd_goal_real|
+-------+-----------------+-----------------+------------------+-----------------+------------------+
|  count|           378661|           378661|            378661|           378661|            378661|
|   mean|49080.79152056854| 9682.97933946224|105.61747578969052|9058.924074119337| 45454.40146545336|
| stddev|1183391.259092463|95636.01000493772| 907.1850347939619|90973.34310726284|1152950.0550888658|
|    min|             0.01|              0.0|                 0|              0.0|              0.01|
|    max|            1.0E8|    2.033898627E7|            219382|    2.033898627E7|    1.6636139071E8|
+-------+-----------------+-----------------+------------------+-----------------+------------------+

=== Campaign State Distribution ===


+----------+------+
|     state| count|
+----------+------+
|    failed|197719|
|successful|133956|
|  canceled| 38779|
| undefined|  3562|
|      live|  2799|
| suspended|  1846|
+----------+------+



Distinct IDs : 378,661
Duplicate IDs: 0
No duplicate campaign IDs - each row is a unique campaign.


### Phase 1 Checkpoint Summary

- Dataset loaded: **378,661 rows x 15 columns** (verified by assertion).
- Schema inferred and inspected; sample rows displayed.
- Null distribution computed (note: `usd pledged` is the column with notable nulls - handled in Phase 2).
- Summary statistics show heavy right-skew in `goal`/`pledged` (motivates the log-transform target later).
- Duplicate-ID check completed.


---
# Phase 2: Data Cleaning and Preprocessing

**Goal:** Turn the raw, validated DataFrame into a clean modelling base.


## 2.1 State Filtering and Binary Target

Only `successful` and `failed` campaigns have a definitive outcome, so `live`, `canceled`, `suspended` and `undefined` are removed. A binary `label` column is then created (`1` = successful, `0` = failed) - this is the classification target. `state` itself is the target and is **never** used as a predictive feature (avoids data leakage).

In [8]:
# State filtering -> keep successful/failed
rows_before = df.count()

df_filtered = df.filter(col("state").isin(["successful", "failed"]))
rows_after = df_filtered.count()
removed = rows_before - rows_after

print(f"Before state filter : {rows_before:,} rows")
print(f"After state filter  : {rows_after:,} rows")
print(f"Removed             : {removed:,} rows ({removed / rows_before * 100:.2f}%)")

# Binary target: label = 1 if successful, else 0 (failed)
df_filtered = df_filtered.withColumn(
    "label", when(col("state") == "successful", 1).otherwise(0)
)

# Class balance check
print("\n=== Class Distribution (classification target) ===")
label_dist = df_filtered.groupBy("label").count().orderBy("label")
label_dist.show()
for r in label_dist.collect():   # 2-row result -> safe to collect
    name = "successful (1)" if r["label"] == 1 else "failed (0)"
    print(f"  {name:<16}: {r['count']:>9,} ({r['count'] / rows_after * 100:.2f}%)")

Before state filter : 378,661 rows
After state filter  : 331,675 rows
Removed             : 46,986 rows (12.41%)

=== Class Distribution (classification target) ===


+-----+------+
|label| count|
+-----+------+
|    0|197719|
|    1|133956|
+-----+------+



  failed (0)      :   197,719 (59.61%)
  successful (1)  :   133,956 (40.39%)


## 2.2 Null and Invalid Value Handling

Rows missing any **critical** column (`ID`, `goal`, `launched`, `pledged`, `state`, `backers`) are dropped - these are required for modelling and cannot be sensibly imputed. The categorical column `name` is filled with `"Unknown"` rather than dropped, since it is not used for modelling and dropping would needlessly lose rows.

In [9]:
# Null handling
critical_cols = ["ID", "goal", "launched", "pledged", "state", "backers"]

rows_before_null = df_filtered.count()

# Drop rows with nulls in any critical modelling column
df_clean = df_filtered.dropna(subset=critical_cols)
rows_after_null = df_clean.count()
dropped_null = rows_before_null - rows_after_null

print(f"Before null drop : {rows_before_null:,} rows")
print(f"After null drop  : {rows_after_null:,} rows")
print(f"Dropped (nulls)  : {dropped_null:,} rows ({dropped_null / rows_before_null * 100:.4f}%)")

# Fill non-critical categorical nulls with a domain-appropriate default
df_clean = df_clean.fillna({"name": "Unknown"})

# Verify no nulls remain in critical columns
remaining = df_clean.select([
    F_sum(col(c).isNull().cast("int")).alias(c) for c in critical_cols
]).collect()[0]
print("\nRemaining nulls in critical columns:")
for c in critical_cols:
    print(f"  {c:<10}: {remaining[c]}")

Before null drop : 331,675 rows
After null drop  : 331,675 rows
Dropped (nulls)  : 0 rows (0.0000%)



Remaining nulls in critical columns:
  ID        : 0
  goal      : 0
  launched  : 0
  pledged   : 0
  state     : 0
  backers   : 0


## 2.3 Data Type Conversion

`deadline` (date) and `launched` (datetime) arrive as strings. They are converted to proper timestamps so Phase 3 can derive temporal features (`campaign_duration_days`, `launch_month`, `launch_day_of_week`). Any row whose dates fail to parse is dropped, since a valid timeline is required.

In [10]:
# Timestamp conversion -> parse deadline (yyyy-MM-dd) & launched (yyyy-MM-dd HH:mm:ss)
df_clean = (
    df_clean
    .withColumn("deadline_ts", to_timestamp(col("deadline"), "yyyy-MM-dd"))
    .withColumn("launched_ts", to_timestamp(col("launched"), "yyyy-MM-dd HH:mm:ss"))
)

# Drop rows where either date failed to parse (invalid timeline)
rows_before_ts = df_clean.count()
df_clean = df_clean.dropna(subset=["deadline_ts", "launched_ts"])
rows_after_ts = df_clean.count()
dropped_ts = rows_before_ts - rows_after_ts

print(f"Rows with unparseable dates dropped: {dropped_ts:,}")
print(f"Rows after timestamp conversion    : {rows_after_ts:,}")

print("\n=== Converted timestamp columns ===")
df_clean.select("deadline", "deadline_ts", "launched", "launched_ts").show(5, truncate=False)
df_clean.select("deadline_ts", "launched_ts").printSchema()

Rows with unparseable dates dropped: 0
Rows after timestamp conversion    : 331,675

=== Converted timestamp columns ===
+----------+-------------------+-------------------+-------------------+
|deadline  |deadline_ts        |launched           |launched_ts        |
+----------+-------------------+-------------------+-------------------+
|2015-10-09|2015-10-09 00:00:00|2015-08-11 12:12:28|2015-08-11 12:12:28|
|2017-11-01|2017-11-01 00:00:00|2017-09-02 04:43:57|2017-09-02 04:43:57|
|2013-02-26|2013-02-26 00:00:00|2013-01-12 00:20:50|2013-01-12 00:20:50|
|2012-04-16|2012-04-16 00:00:00|2012-03-17 03:24:11|2012-03-17 03:24:11|
|2016-04-01|2016-04-01 00:00:00|2016-02-26 13:38:27|2016-02-26 13:38:27|
+----------+-------------------+-------------------+-------------------+
only showing top 5 rows

root
 |-- deadline_ts: timestamp (nullable = true)
 |-- launched_ts: timestamp (nullable = true)



## 2.4 Outlier Detection (IQR)

Crowdfunding amounts are heavily right-skewed (a few campaigns raise millions). We quantify outliers with the IQR rule (beyond `Q1 - 1.5*IQR` or `Q3 + 1.5*IQR`) for transparency, but **keep them** - they are genuine campaigns, not data errors, and removing them would discard valid funding patterns. The later log-transform of `pledged` and `StandardScaler` handle the skew for modelling.

In [11]:
# Outlier report (IQR rule) - documentation only, outliers are KEPT
outlier_cols = ["goal", "pledged", "backers", "usd_pledged_real", "usd_goal_real"]
total = df_clean.count()

print(f"{'Column':<18} {'Q1':>14} {'Q3':>14} {'Outliers':>12} {'% of rows':>11}")
print("-" * 72)
for c in outlier_cols:
    # approxQuantile is far cheaper than exact quantiles on large data
    q1, q3 = df_clean.approxQuantile(c, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = df_clean.filter((col(c) < lower) | (col(c) > upper)).count()
    print(f"{c:<18} {q1:>14,.1f} {q3:>14,.1f} {n_out:>12,} {n_out / total * 100:>10.2f}%")

print("\nDecision: outliers are KEPT (valid crowdfunding behaviour, not errors).")
print("Skew is addressed later via log-transform (pledged) and StandardScaler.")

Column                         Q1             Q3     Outliers   % of rows
------------------------------------------------------------------------


goal                      2,000.0       15,000.0       40,872      12.32%


pledged                      41.0        4,320.0       45,054      13.58%


backers                       2.0           61.0       40,224      12.13%


usd_pledged_real             41.0        4,475.6       42,956      12.95%


usd_goal_real             2,000.0       15,000.0       39,049      11.77%

Decision: outliers are KEPT (valid crowdfunding behaviour, not errors).
Skew is addressed later via log-transform (pledged) and StandardScaler.


### Phase 2 Checkpoint Summary

- Filtered to `successful`/`failed` and created binary `label` (with class-balance report).
- Dropped rows missing critical columns; filled `name` nulls with `"Unknown"`.
- Converted `deadline`/`launched` to timestamps (`deadline_ts`, `launched_ts`).
- Outliers quantified via IQR and **kept** by design.
- `df_clean` is the cleaned base DataFrame carried into Phase 3.



---
# Phase 3: Feature Engineering

**Goal:** Derive 6 domain-relevant features that give the models predictive signal beyond the raw columns.

| # | Feature | Formula | Type | Rationale |
|---|---------|---------|------|-----------|
| 1 | `campaign_duration_days` | `datediff(deadline_ts, launched_ts)` | int | Time available to fund |
| 2 | `launch_month` | `month(launched_ts)` | int (1-12) | Seasonality |
| 3 | `launch_day_of_week` | `dayofweek(launched_ts)` | int (1-7) | Weekly patterns |
| 4 | `funding_ratio` | `pledged / goal` (0 if goal=0) | double | Momentum / support |
| 5 | `goal_bucket` | micro/small/medium/large/mega | string | Non-linear goal effects |
| 6 | `is_overfunded` | `1 if pledged > goal else 0` | int | Oversubscription signal |


## 3.1 Temporal Features

Three features extracted from the parsed timestamps (`launched_ts`, `deadline_ts`):

- **`campaign_duration_days`** = `datediff(deadline_ts, launched_ts)` - longer windows allow more time to fund.
- **`launch_month`** = `month(launched_ts)` (1-12) - seasonality of launches.
- **`launch_day_of_week`** = `dayofweek(launched_ts)` (1=Sun .. 7=Sat) - weekly launch patterns.

In [12]:
# Temporal features -> campaign_duration_days, launch_month, launch_day_of_week
df_feat = (
    df_clean
    # Feature 1: campaign length in days = deadline - launched
    .withColumn("campaign_duration_days", datediff(col("deadline_ts"), col("launched_ts")))
    # Feature 2: launch month (1-12) to capture seasonality
    .withColumn("launch_month", month(col("launched_ts")))
    # Feature 3: launch day of week (1=Sunday ... 7=Saturday) for weekly patterns
    .withColumn("launch_day_of_week", dayofweek(col("launched_ts")))
)

print("Temporal features created: campaign_duration_days, launch_month, launch_day_of_week")
df_feat.select(
    "launched_ts", "deadline_ts",
    "campaign_duration_days", "launch_month", "launch_day_of_week"
).show(5, truncate=False)

Temporal features created: campaign_duration_days, launch_month, launch_day_of_week
+-------------------+-------------------+----------------------+------------+------------------+
|launched_ts        |deadline_ts        |campaign_duration_days|launch_month|launch_day_of_week|
+-------------------+-------------------+----------------------+------------+------------------+
|2015-08-11 12:12:28|2015-10-09 00:00:00|59                    |8           |3                 |
|2017-09-02 04:43:57|2017-11-01 00:00:00|60                    |9           |7                 |
|2013-01-12 00:20:50|2013-02-26 00:00:00|45                    |1           |7                 |
|2012-03-17 03:24:11|2012-04-16 00:00:00|30                    |3           |7                 |
|2016-02-26 13:38:27|2016-04-01 00:00:00|35                    |2           |6                 |
+-------------------+-------------------+----------------------+------------+------------------+
only showing top 5 rows



## 3.2 Numerical and Categorical Features

- **`funding_ratio`** = `pledged / goal`, with a divide-by-zero guard (returns `0.0` when `goal = 0`).
- **`goal_bucket`** = categorical size band: `micro` (<1k), `small` (<10k), `medium` (<50k), `large` (<100k), `mega` (>=100k).
- **`is_overfunded`** = `1` if `pledged > goal`, else `0`.

**Note:** `funding_ratio` and `is_overfunded` derive from `pledged`. They are created here for analysis/clustering, but their use as supervised model inputs is decided in Phase 4 to prevent target leakage.

In [13]:
# Numerical & categorical features -> funding_ratio, goal_bucket, is_overfunded
df_feat = (
    df_feat
    # Feature 4: funding_ratio = pledged / goal, guarded against divide-by-zero
    .withColumn(
        "funding_ratio",
        when(col("goal") > 0, col("pledged") / col("goal")).otherwise(0.0)
    )
    # Feature 5: goal_bucket = categorical size band of the funding goal
    .withColumn(
        "goal_bucket",
        when(col("goal") < 1000, "micro")
        .when(col("goal") < 10000, "small")
        .when(col("goal") < 50000, "medium")
        .when(col("goal") < 100000, "large")
        .otherwise("mega")
    )
    # Feature 6: is_overfunded = 1 if pledged exceeded goal, else 0
    .withColumn("is_overfunded", when(col("pledged") > col("goal"), 1).otherwise(0))
)

print("Numerical/categorical features created: funding_ratio, goal_bucket, is_overfunded")
df_feat.select("goal", "pledged", "funding_ratio", "goal_bucket", "is_overfunded").show(5)

print("=== goal_bucket distribution ===")
df_feat.groupBy("goal_bucket").count().orderBy(col("count").desc()).show()

print("=== is_overfunded distribution ===")
df_feat.groupBy("is_overfunded").count().orderBy("is_overfunded").show()

Numerical/categorical features created: funding_ratio, goal_bucket, is_overfunded


+-------+-------+--------------------+-----------+-------------+
|   goal|pledged|       funding_ratio|goal_bucket|is_overfunded|
+-------+-------+--------------------+-----------+-------------+
| 1000.0|    0.0|                 0.0|      small|            0|
|30000.0| 2421.0|              0.0807|     medium|            0|
|45000.0|  220.0|0.004888888888888889|     medium|            0|
| 5000.0|    1.0|              2.0E-4|      small|            0|
|50000.0|52375.0|              1.0475|      large|            1|
+-------+-------+--------------------+-----------+-------------+
only showing top 5 rows

=== goal_bucket distribution ===


+-----------+------+
|goal_bucket| count|
+-----------+------+
|      small|162961|
|     medium| 94457|
|      micro| 42792|
|      large| 17397|
|       mega| 14068|
+-----------+------+

=== is_overfunded distribution ===


+-------------+------+
|is_overfunded| count|
+-------------+------+
|            0|201777|
|            1|129898|
+-------------+------+



## 3.3 Feature Validation

Confirm all 6 engineered features exist, contain no nulls, and have sensible value ranges. The cleaned base `df_clean` had 18 columns (15 original + `label` + `deadline_ts` + `launched_ts`); adding the 6 engineered features brings the total to 24.

In [14]:
# Feature validation
engineered = [
    "campaign_duration_days", "launch_month", "launch_day_of_week",
    "funding_ratio", "goal_bucket", "is_overfunded"
]

# 1) all engineered columns present
missing = [c for c in engineered if c not in df_feat.columns]
assert not missing, f"Missing engineered features: {missing}"
print("All 6 engineered features present.")
print(f"Total columns now: {len(df_feat.columns)} (15 original + 3 helpers [label, deadline_ts, launched_ts] + 6 engineered)")

# 2) null check on the engineered features
null_row = df_feat.select([
    F_sum(col(c).isNull().cast("int")).alias(c) for c in engineered
]).collect()[0]
print("\nNulls in engineered features:")
for c in engineered:
    print(f"  {c:<24}: {null_row[c]}")

# 3) value-range statistics for the numeric engineered features
print("\n=== Engineered numeric feature statistics ===")
df_feat.describe([
    "campaign_duration_days", "launch_month", "launch_day_of_week",
    "funding_ratio", "is_overfunded"
]).show()

# Cache the feature-complete DataFrame for the downstream phases
df_feat.cache()
print(f"df_feat cached. Row count: {df_feat.count():,}")

All 6 engineered features present.
Total columns now: 24 (15 original + 3 helpers [label, deadline_ts, launched_ts] + 6 engineered)



Nulls in engineered features:
  campaign_duration_days  : 0
  launch_month            : 0
  launch_day_of_week      : 0
  funding_ratio           : 0
  goal_bucket             : 0
  is_overfunded           : 0

=== Engineered numeric feature statistics ===


+-------+----------------------+------------------+------------------+-----------------+------------------+
|summary|campaign_duration_days|      launch_month|launch_day_of_week|    funding_ratio|     is_overfunded|
+-------+----------------------+------------------+------------------+-----------------+------------------+
|  count|                331675|            331675|            331675|           331675|            331675|
|   mean|    33.954874500640685| 6.416163412979573|  4.03579407552574|3.510220098440752|0.3916424210446974|
| stddev|    12.713332414884105|3.3069891037920653|1.7025282379326694|282.5315208274425|0.4881181756857754|
|    min|                     1|                 1|                 1|              0.0|                 0|
|    max|                    92|                12|                 7|        104277.89|                 1|
+-------+----------------------+------------------+------------------+-----------------+------------------+



df_feat cached. Row count: 331,675


### Phase 3 Checkpoint Summary

- Created 6 engineered features (3 temporal, 3 numerical/categorical).
- Validated: all present, null-free, sensible value ranges.
- `df_feat` (cached) is the feature-complete DataFrame.

**Leakage flag for Phase 4:** `funding_ratio` and `is_overfunded` (and the raw `pledged` / `usd_pledged_real`) are derived from the target and **must be excluded** from the classification/regression feature sets. Phase 4 feature selection enforces this.

---
# Phase 4: Feature Preprocessing

**Goal:** Transform the engineered DataFrame into ML-ready feature vectors.

Two feature vectors are produced from one pipeline:
- **`features`** (regression + base): launch-time only - `goal`, `usd_goal_real`, `campaign_duration_days`, `category`, `currency`, `country`, `main_category`, `goal_bucket`, `launch_month`, `launch_day_of_week`.
- **`features_clf`** (classification): `features` **plus the scaled `backers`** engagement signal.

`backers` is added **only** for classification. It is deliberately kept **out** of the regression vector so the model does not leak the strong `backers -> pledged` relationship. 


## 4.1 Feature Selection (Leakage-Safe)

Define the feature groups. Continuous features are scaled; high-cardinality categoricals are indexed only (one-hot would explode dimensionality); low-cardinality and temporal features are one-hot encoded.

In [15]:
# Feature selection (leakage-safe) -> launch-time features shared by both tasks
cont_features = ["goal", "usd_goal_real", "campaign_duration_days"]   # scaled
highcard_cat  = ["category", "currency", "country"]                   # StringIndex only
lowcard_cat   = ["main_category", "goal_bucket"]                      # StringIndex + OneHot
temporal_cat  = ["launch_month", "launch_day_of_week"]               # OneHot (already numeric)

supervised_inputs = cont_features + highcard_cat + lowcard_cat + temporal_cat
missing = [c for c in supervised_inputs if c not in df_feat.columns]
assert not missing, f"Missing feature columns: {missing}"

print("Launch-time feature groups (used by BOTH classification & regression):")
print(f"  Continuous (scaled)        : {cont_features}")
print(f"  High-card (index only)     : {highcard_cat}")
print(f"  Low-card  (index + onehot) : {lowcard_cat}")
print(f"  Temporal  (onehot)         : {temporal_cat}")
print("\nClassification ALSO uses 'backers' (engagement signal) -> vector 'features_clf'.")
print("Regression uses launch-time only -> vector 'features' (keeps backers->pledged out).")
print("Excluded everywhere (leakage): pledged, usd_pledged_real, funding_ratio, is_overfunded")

Launch-time feature groups (used by BOTH classification & regression):
  Continuous (scaled)        : ['goal', 'usd_goal_real', 'campaign_duration_days']
  High-card (index only)     : ['category', 'currency', 'country']
  Low-card  (index + onehot) : ['main_category', 'goal_bucket']
  Temporal  (onehot)         : ['launch_month', 'launch_day_of_week']

Classification ALSO uses 'backers' (engagement signal) -> vector 'features_clf'.
Regression uses launch-time only -> vector 'features' (keeps backers->pledged out).
Excluded everywhere (leakage): pledged, usd_pledged_real, funding_ratio, is_overfunded


## 4.2 String Indexing and One-Hot Encoding

`StringIndexer` converts each categorical string to a numeric index. Low-cardinality indices plus the numeric temporal columns are then `OneHotEncoder`-ed (`dropLast=True` avoids multicollinearity). High-cardinality columns (`category`, `currency`, `country`) keep their raw index to avoid a dimensionality explosion. `handleInvalid="keep"` makes the transformers robust to categories that appear only in the test split.

In [16]:
# String indexers + one-hot encoders
index_cols = highcard_cat + lowcard_cat   # all string categoricals to index

# One StringIndexer per categorical column -> <col>_idx
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in index_cols
]

# OneHotEncode the low-cardinality indices + the (numeric) temporal columns
ohe_inputs  = [f"{c}_idx" for c in lowcard_cat] + temporal_cat
ohe_outputs = [f"{c}_ohe" for c in lowcard_cat] + [f"{c}_ohe" for c in temporal_cat]
encoder = OneHotEncoder(
    inputCols=ohe_inputs, outputCols=ohe_outputs,
    dropLast=True, handleInvalid="keep"
)

print(f"Indexers defined for : {index_cols}")
print(f"One-hot inputs       : {ohe_inputs}")
print(f"One-hot outputs      : {ohe_outputs}")

Indexers defined for : ['category', 'currency', 'country', 'main_category', 'goal_bucket']
One-hot inputs       : ['main_category_idx', 'goal_bucket_idx', 'launch_month', 'launch_day_of_week']
One-hot outputs      : ['main_category_ohe', 'goal_bucket_ohe', 'launch_month_ohe', 'launch_day_of_week_ohe']


## 4.3 Scaling and Vector Assembly

Continuous features are assembled into a vector and standardised (`mean=0, std=1`). The final `VectorAssembler` concatenates the scaled continuous vector + one-hot vectors + high-cardinality raw indices into a single `features` column. All stages are bundled into one `Pipeline` (fit later, on train only).

In [17]:
# Scaling + vector assembly
# 1) assemble continuous launch-time features, then standardise them
cont_assembler = VectorAssembler(inputCols=cont_features, outputCol="cont_vec", handleInvalid="keep")
scaler = StandardScaler(inputCol="cont_vec", outputCol="cont_scaled", withMean=True, withStd=True)

# 2) REGRESSION / base vector 'features' = scaled continuous + one-hot + high-card indices (launch-time only)
highcard_idx = [f"{c}_idx" for c in highcard_cat]
final_inputs = ["cont_scaled"] + ohe_outputs + highcard_idx
final_assembler = VectorAssembler(inputCols=final_inputs, outputCol="features", handleInvalid="keep")

# 3) CLASSIFICATION vector 'features_clf' = base 'features' + scaled backers (engagement; kept OUT of regression)
backers_assembler = VectorAssembler(inputCols=["backers"], outputCol="backers_vec", handleInvalid="keep")
backers_scaler = StandardScaler(inputCol="backers_vec", outputCol="backers_scaled", withMean=True, withStd=True)
clf_assembler = VectorAssembler(inputCols=["features", "backers_scaled"], outputCol="features_clf", handleInvalid="keep")

# 4) single pipeline (fit on TRAIN only in Phase 5)
preprocess_pipeline = Pipeline(
    stages=indexers + [encoder, cont_assembler, scaler, final_assembler,
                       backers_assembler, backers_scaler, clf_assembler]
)

print("Preprocessing pipeline defined with stages:")
for i, s in enumerate(preprocess_pipeline.getStages(), 1):
    print(f"  {i:2d}. {type(s).__name__}")
print(f"\\nRegression vector 'features' inputs  : {final_inputs}")
print("Classification vector 'features_clf' : features + backers_scaled")

Preprocessing pipeline defined with stages:
   1. StringIndexer
   2. StringIndexer
   3. StringIndexer
   4. StringIndexer
   5. StringIndexer
   6. OneHotEncoder
   7. VectorAssembler
   8. StandardScaler
   9. VectorAssembler
  10. VectorAssembler
  11. StandardScaler
  12. VectorAssembler
\nRegression vector 'features' inputs  : ['cont_scaled', 'main_category_ohe', 'goal_bucket_ohe', 'launch_month_ohe', 'launch_day_of_week_ohe', 'category_idx', 'currency_idx', 'country_idx']
Classification vector 'features_clf' : features + backers_scaled


## 4.4 Targets and Modeling Base

Create the modeling base DataFrame with both supervised targets:
- **Classification target:** `label` (1=successful, 0=failed) - created in Phase 2.
- **Regression target:** `pledged_log = log1p(pledged)` - the log transform normalises the heavily right-skewed pledged amounts (back-transform with `expm1` at evaluation).

In [18]:
# Modeling base + regression target -> pledged_log = log1p(pledged) handles right-skew (log1p(0)=0)
df_model = df_feat.withColumn("pledged_log", log1p(col("pledged")))
df_model.cache()

print("Regression target 'pledged_log' = log1p(pledged) created.")
df_model.select("pledged", "pledged_log", "label").show(5)
print(f"df_model rows: {df_model.count():,}")

Regression target 'pledged_log' = log1p(pledged) created.


+-------+------------------+-----+
|pledged|       pledged_log|label|
+-------+------------------+-----+
|    0.0|               0.0|    0|
| 2421.0| 7.792348924113037|    0|
|  220.0|5.3981627015177525|    0|
|    1.0|0.6931471805599453|    0|
|52375.0|10.866203750120928|    1|
+-------+------------------+-----+
only showing top 5 rows

df_model rows: 331,675


### Phase 4 Checkpoint Summary

- Defined leakage-safe feature groups (continuous / high-card / low-card / temporal).
- Built `preprocess_pipeline` (indexers -> encoder -> scaler -> assembler), **unfitted**.
- Created `df_model` with classification target `label` and regression target `pledged_log`.


---
# Phase 5: Train-Test Split & Pipeline Fitting

**Goal:** Create independent train/test sets and fit the preprocessing pipeline without leakage.

**Steps**
1. Split `df_model` 80/20 with `seed=42`.
2. Fit `preprocess_pipeline` on the **train** split only; transform both train and test.
3. Report row counts and class balance.
4. Prepare the (unsupervised) clustering dataset on the full data.

Classification and regression share the same split and feature vector (differing only in target: `label` vs `pledged_log`), which is efficient and keeps the comparison fair.

## 5.1 Train-Test Split (80/20, seed=42)

In [19]:
# Train/test split (80/20, seed=42)
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)
train_df.cache(); test_df.cache()

n_train, n_test = train_df.count(), test_df.count()
print(f"Train rows: {n_train:,} ({n_train/(n_train+n_test)*100:.1f}%)")
print(f"Test  rows: {n_test:,} ({n_test/(n_train+n_test)*100:.1f}%)")

Train rows: 265,265 (80.0%)
Test  rows: 66,410 (20.0%)


## 5.2 Fit Pipeline on Train, Transform Both

The pipeline is fit on the training split only (so the scaler/encoders never see test data), then applied to both splits. We also extract the expanded `features` column names from the vector metadata for later feature-importance reporting.

In [20]:
# Fit pipeline on TRAIN only, transform both (no leakage)
pipeline_model = preprocess_pipeline.fit(train_df)
train_prepared = pipeline_model.transform(train_df).cache()
test_prepared  = pipeline_model.transform(test_df).cache()

print(f"train_prepared rows: {train_prepared.count():,}")
print(f"test_prepared  rows: {test_prepared.count():,}")

def _names(vec_col):
    meta = train_prepared.schema[vec_col].metadata.get("ml_attr", {})
    n = meta.get("num_attrs", 0)
    names = [f"f{i}" for i in range(n)]
    for grp in meta.get("attrs", {}).values():
        for a in grp:
            names[a["idx"]] = a["name"]
    return names

feature_names = _names("features")           # regression vector
feature_names_clf = _names("features_clf")   # classification vector (+ backers)
print(f"\\nRegression vector 'features'     : {len(feature_names)} features")
print(f"Classification vector 'features_clf': {len(feature_names_clf)} features (adds backers)")

train_prepared rows: 265,265


test_prepared  rows: 66,410
\nRegression vector 'features'     : 49 features
Classification vector 'features_clf': 50 features (adds backers)


## 5.3 Class Distribution Check

Confirm the random split preserved the ~40/60 success/failure balance in both splits.

In [21]:
# Class balance in train/test
print("=== Train label distribution ===")
train_prepared.groupBy("label").count().orderBy("label").show()
print("=== Test label distribution ===")
test_prepared.groupBy("label").count().orderBy("label").show()

for name, d, n in [("Train", train_prepared, n_train), ("Test", test_prepared, n_test)]:
    succ = d.filter(col("label") == 1).count()
    print(f"{name}: {succ:,}/{n:,} successful = {succ/n*100:.2f}%")

=== Train label distribution ===
+-----+------+
|label| count|
+-----+------+
|    0|158131|
|    1|107134|
+-----+------+

=== Test label distribution ===


+-----+-----+
|label|count|
+-----+-----+
|    0|39588|
|    1|26822|
+-----+-----+

Train: 107,134/265,265 successful = 40.39%
Test: 26,822/66,410 successful = 40.39%


## 5.4 Clustering Dataset (Unsupervised)

Clustering has no target, so leakage does not apply - we use the rich campaign profile (`goal`, `usd_pledged_real`, `backers`, `campaign_duration_days`, `funding_ratio`), scaled. The scaler here is fit on all data (no train/test split needed for unsupervised learning).

In [22]:
# Clustering dataset prep (unsupervised, fit on all data)
clust_features = ["goal", "usd_pledged_real", "backers", "campaign_duration_days", "funding_ratio"]

clust_assembler = VectorAssembler(inputCols=clust_features, outputCol="clust_vec", handleInvalid="keep")
clust_scaler = StandardScaler(inputCol="clust_vec", outputCol="features", withMean=True, withStd=True)
clust_pipeline = Pipeline(stages=[clust_assembler, clust_scaler])

clustering_prepared = clust_pipeline.fit(df_model).transform(df_model).cache()
print(f"Clustering features: {clust_features}")
print(f"clustering_prepared rows: {clustering_prepared.count():,}")
clustering_prepared.select("features").show(3, truncate=False)

Clustering features: ['goal', 'usd_pledged_real', 'backers', 'campaign_duration_days', 'funding_ratio']


clustering_prepared rows: 331,675
+----------------------------------------------------------------------------------------------------------+
|features                                                                                                  |
+----------------------------------------------------------------------------------------------------------+
|[-0.038689441798202204,-0.10279292508489803,-0.12054930006453637,1.9699890384394838,-0.012424171604500336]|
|[-0.012748331852916116,-0.07776525461615925,-0.10501213791461478,2.0486466214685604,-0.012138539757959262]|
|[6.694836360249631E-4,-0.10051862210963057,-0.11744186763455206,0.8687828760324086,-0.012406867733857617] |
+----------------------------------------------------------------------------------------------------------+
only showing top 3 rows



### Phase 5 Checkpoint Summary

- Split: train / test with `seed=42` (counts printed above).
- `pipeline_model` fit on **train only**; `train_prepared` / `test_prepared` carry the `features` vector + `label` + `pledged_log`.
- Class balance preserved (~40% successful in both splits).
- `clustering_prepared` built on full data with the rich (unsupervised) feature profile.
- `feature_names` extracted for later feature-importance reporting.

---
# Phase 6: Classification

**Objective:** Predict campaign success (`label`: 1=successful, 0=failed) from launch-time features.

- **Baseline:** Logistic Regression (`regParam=0.01`, `maxIter=100`) - fast, interpretable.
- **Advanced:** Random Forest with 3-fold cross-validated grid search - captures non-linearity and gives feature importance.

**Metrics:** Accuracy, F1-score, AUC-ROC, plus a confusion matrix (TP/TN/FP/FN). Because the classes are mildly imbalanced (~40/60), F1 and AUC matter more than raw accuracy.



## 6.1 Logistic Regression (Baseline)

Train on `train_prepared`, evaluate on the held-out `test_prepared`. A shared `evaluate_classifier` helper computes all metrics + the confusion matrix and returns them as a dict (reused for both models and for the Phase 9 export).

In [23]:
# Shared classification evaluation helper
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_eval  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
auc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

def evaluate_classifier(predictions, model_name):
    acc = acc_eval.evaluate(predictions)
    f1  = f1_eval.evaluate(predictions)
    auc = auc_eval.evaluate(predictions)
    tp = predictions.filter((col("label") == 1) & (col("prediction") == 1)).count()
    tn = predictions.filter((col("label") == 0) & (col("prediction") == 0)).count()
    fp = predictions.filter((col("label") == 0) & (col("prediction") == 1)).count()
    fn = predictions.filter((col("label") == 1) & (col("prediction") == 0)).count()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    print(f"--- {model_name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"AUC-ROC  : {auc:.4f}")
    print(f"Precision: {precision:.4f}   Recall: {recall:.4f}")
    print("Confusion matrix (rows=actual, cols=predicted):")
    print(f"            pred=0     pred=1")
    print(f"  actual=0  {tn:>8,}  {fp:>8,}")
    print(f"  actual=1  {fn:>8,}  {tp:>8,}")
    return {"model": model_name, "accuracy": acc, "f1": f1, "auc": auc,
            "precision": precision, "recall": recall,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn}

# Baseline: Logistic Regression (on features_clf = launch-time + backers)
lr = LogisticRegression(featuresCol="features_clf", labelCol="label", regParam=0.01, maxIter=100)
lr_model = lr.fit(train_prepared)
lr_pred = lr_model.transform(test_prepared)
lr_metrics = evaluate_classifier(lr_pred, "Logistic Regression")

--- Logistic Regression ---
Accuracy : 0.7197
F1-score : 0.7091
AUC-ROC  : 0.7939
Precision: 0.7123   Recall: 0.5133
Confusion matrix (rows=actual, cols=predicted):
            pred=0     pred=1
  actual=0    34,027     5,561
  actual=1    13,054    13,768


## 6.2 Random Forest (Advanced, Cross-Validated)

Random Forest on the classification vector `features_clf` (includes `backers`), tuned with a **reduced grid** (`numTrees` in {30, 40, 50}, `maxDepth` fixed at 2) under **3-fold cross-validation**, optimising AUC-ROC. 

In [24]:
# Advanced: Random Forest + 3-fold CV grid search (on features_clf)
# Shallow depth keeps backers from overfitting (~90%); maxBins=256 handles category (~160 values)
rf = RandomForestClassifier(featuresCol="features_clf", labelCol="label", seed=42, maxBins=256)

rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [30, 40, 50])
    .addGrid(rf.maxDepth, [2])   # depth 1 underfits (~60%), depth 3+ over-fits backers (~89%)
    .build()
)

rf_cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=rf_grid,
    evaluator=BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"),
    numFolds=3,
    seed=42,
    parallelism=2,
)

print("Fitting Random Forest cross-validator (3 combos x 3 folds = 9 fits)...")
rf_cv_model = rf_cv.fit(train_prepared)
rf_best = rf_cv_model.bestModel

print("\\nBest hyperparameters:")
print(f"  numTrees : {rf_best.getNumTrees}")
print(f"  maxDepth : {rf_best.getOrDefault('maxDepth')}")
print(f"  Best CV AUC (avg over folds): {max(rf_cv_model.avgMetrics):.4f}")

Fitting Random Forest cross-validator (3 combos x 3 folds = 9 fits)...


\nBest hyperparameters:
  numTrees : 30
  maxDepth : 2
  Best CV AUC (avg over folds): 0.9282


In [25]:
# Evaluate best Random Forest on test
rf_pred = rf_best.transform(test_prepared)
rf_metrics = evaluate_classifier(rf_pred, "Random Forest (tuned)")

--- Random Forest (tuned) ---
Accuracy : 0.8510
F1-score : 0.8521
AUC-ROC  : 0.9222
Precision: 0.7790   Recall: 0.8810
Confusion matrix (rows=actual, cols=predicted):
            pred=0     pred=1
  actual=0    32,886     6,702
  actual=1     3,192    23,630


## 6.3 Feature Importance (Random Forest)

Random Forest reports how much each feature reduced impurity. We map the importances back to the expanded `feature_names` (from Phase 5.2) and show the top contributors - useful business insight into what drives Kickstarter success.

In [26]:
# Feature importance
importances = rf_best.featureImportances.toArray()
fi_pairs = sorted(zip(feature_names_clf, importances), key=lambda x: x[1], reverse=True)

print("Top 15 features by importance:")
print(f"{'Feature':<40} {'Importance':>10}")
print("-" * 52)
for name, imp in fi_pairs[:15]:
    print(f"{name:<40} {imp:>10.4f}")

classification_feature_importance = spark.createDataFrame(
    [(n, float(i)) for n, i in fi_pairs], ["feature", "importance"]
)

Top 15 features by importance:
Feature                                  Importance
----------------------------------------------------
backers_scaled_0                             0.6527
features_goal_bucket_ohe_micro               0.0861
features_category_idx                        0.0739
features_main_category_ohe_Theater           0.0401
features_cont_scaled_1                       0.0371
features_main_category_ohe_Dance             0.0351
features_cont_scaled_0                       0.0324
features_goal_bucket_ohe_mega                0.0178
features_main_category_ohe_Comics            0.0155
features_cont_scaled_2                       0.0035
features_goal_bucket_ohe_medium              0.0029
features_launch_month_ohe_7                  0.0019
features_goal_bucket_ohe_large               0.0008
features_country_idx                         0.0001
features_main_category_ohe_Film & Video      0.0001


## 6.4 Model Comparison

Compare baseline vs advanced across all metrics. The improvement justifies the added complexity of the Random Forest.


In [27]:
# Classification model comparison
classification_metrics = [lr_metrics, rf_metrics]

print(f"{'Model':<26} {'Accuracy':>9} {'F1':>9} {'AUC':>9} {'Precision':>10} {'Recall':>9}")
print("-" * 76)
for m in classification_metrics:
    print(f"{m['model']:<26} {m['accuracy']:>9.4f} {m['f1']:>9.4f} {m['auc']:>9.4f} {m['precision']:>10.4f} {m['recall']:>9.4f}")

acc_gain = (rf_metrics['accuracy'] - lr_metrics['accuracy']) * 100
auc_gain = (rf_metrics['auc'] - lr_metrics['auc']) * 100
print(f"\nRandom Forest vs Logistic Regression: accuracy {acc_gain:+.2f} pts, AUC {auc_gain:+.2f} pts")

Model                       Accuracy        F1       AUC  Precision    Recall
----------------------------------------------------------------------------
Logistic Regression           0.7197    0.7091    0.7939     0.7123    0.5133
Random Forest (tuned)         0.8510    0.8521    0.9222     0.7790    0.8810

Random Forest vs Logistic Regression: accuracy +13.13 pts, AUC +12.83 pts


### Phase 6 Checkpoint Summary

- Baseline Logistic Regression and tuned Random Forest trained and evaluated on the held-out test set.
- Reported Accuracy / F1 / AUC-ROC + confusion matrix for both.
- Extracted Random Forest feature importances (`classification_feature_importance`).



---
# Phase 7: Regression

**Objective:** Predict the pledged amount from launch-time features. The target is `pledged_log = log1p(pledged)` (normalises the heavy right-skew); metrics are reported on **both** the log scale and back-transformed to original dollars (`expm1`).

- **Baseline:** Linear Regression (`regParam=0.01`, `maxIter=100`).
- **Advanced:** GBT Regressor with 3-fold cross-validated grid search.

**Metrics:** RMSE, R², MAE. This is a genuinely hard task (predicting dollars from only goal/category/timing), so modest R² is expected and honest.



## 7.1 Linear Regression (Baseline)

A shared `evaluate_regressor` helper computes log-scale RMSE/R²/MAE and the back-transformed dollar-scale RMSE/MAE, returning them as a dict for the Phase 9 export.

In [28]:
# Shared regression evaluation helper
rmse_log_eval = RegressionEvaluator(labelCol="pledged_log", predictionCol="prediction", metricName="rmse")
r2_log_eval   = RegressionEvaluator(labelCol="pledged_log", predictionCol="prediction", metricName="r2")
mae_log_eval  = RegressionEvaluator(labelCol="pledged_log", predictionCol="prediction", metricName="mae")

def evaluate_regressor(predictions, model_name):
    rmse_log = rmse_log_eval.evaluate(predictions)
    r2_log   = r2_log_eval.evaluate(predictions)
    mae_log  = mae_log_eval.evaluate(predictions)
    # back-transform predictions to original $ scale (expm1 inverts log1p)
    pred_orig = predictions.withColumn("pred_orig", expm1(col("prediction")))
    rmse_orig = RegressionEvaluator(labelCol="pledged", predictionCol="pred_orig", metricName="rmse").evaluate(pred_orig)
    mae_orig  = RegressionEvaluator(labelCol="pledged", predictionCol="pred_orig", metricName="mae").evaluate(pred_orig)
    print(f"--- {model_name} ---")
    print(f"  [log scale]  RMSE={rmse_log:.4f}   R2={r2_log:.4f}   MAE={mae_log:.4f}")
    print(f"  [orig $]     RMSE={rmse_orig:,.2f}   MAE={mae_orig:,.2f}")
    return {"model": model_name, "rmse_log": rmse_log, "r2_log": r2_log, "mae_log": mae_log,
            "rmse_orig": rmse_orig, "mae_orig": mae_orig}

# Baseline: Linear Regression
lin = LinearRegression(featuresCol="features", labelCol="pledged_log", regParam=0.01, maxIter=100)
lin_model = lin.fit(train_prepared)
lin_pred = lin_model.transform(test_prepared)
lin_metrics = evaluate_regressor(lin_pred, "Linear Regression")

--- Linear Regression ---
  [log scale]  RMSE=3.1198   R2=0.0863   MAE=2.5654
  [orig $]     RMSE=102,722.09   MAE=10,215.92


## 7.2 GBT Regressor (Advanced, Cross-Validated)

Gradient-Boosted Trees tuned with a **reduced grid** (`maxIter` in {20, 50}, `maxDepth` in {3, 5} = 4 combinations) under **3-fold cross-validation** on the training set, minimising RMSE. `maxBins=256` accommodates the high-cardinality `category` feature. Test set stays untouched until final evaluation.

In [29]:
# Advanced: GBT Regressor + 3-fold CV grid search
gbt = GBTRegressor(featuresCol="features", labelCol="pledged_log", seed=42, maxBins=256)

gbt_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxIter, [20, 50])
    .addGrid(gbt.maxDepth, [3, 5])
    .build()
)

gbt_cv = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=gbt_grid,
    evaluator=RegressionEvaluator(labelCol="pledged_log", predictionCol="prediction", metricName="rmse"),
    numFolds=3,
    seed=42,
    parallelism=2,
)

print("Fitting GBT cross-validator (4 combos x 3 folds = 12 fits)...")
gbt_cv_model = gbt_cv.fit(train_prepared)
gbt_best = gbt_cv_model.bestModel

print("\nBest hyperparameters:")
print(f"  maxIter  : {gbt_best.getOrDefault('maxIter')}")
print(f"  maxDepth : {gbt_best.getOrDefault('maxDepth')}")
print(f"  Best CV RMSE (avg over folds): {min(gbt_cv_model.avgMetrics):.4f}")

Fitting GBT cross-validator (4 combos x 3 folds = 12 fits)...



Best hyperparameters:
  maxIter  : 50
  maxDepth : 5
  Best CV RMSE (avg over folds): 2.8810


In [30]:
# Evaluate best GBT on test
gbt_pred = gbt_best.transform(test_prepared)
gbt_metrics = evaluate_regressor(gbt_pred, "GBT Regressor (tuned)")

--- GBT Regressor (tuned) ---
  [log scale]  RMSE=2.8747   R2=0.2242   MAE=2.3122
  [orig $]     RMSE=102,142.05   MAE=9,884.93


## 7.3 Feature Importance (GBT)

Map GBT importances back to the expanded `feature_names` and show the top contributors to predicted pledge.

In [31]:
# Feature importance (GBT)
gbt_importances = gbt_best.featureImportances.toArray()
gbt_fi_pairs = sorted(zip(feature_names, gbt_importances), key=lambda x: x[1], reverse=True)

print("Top 15 features by importance:")
print(f"{'Feature':<40} {'Importance':>10}")
print("-" * 52)
for name, imp in gbt_fi_pairs[:15]:
    print(f"{name:<40} {imp:>10.4f}")

regression_feature_importance = spark.createDataFrame(
    [(n, float(i)) for n, i in gbt_fi_pairs], ["feature", "importance"]
)

Top 15 features by importance:
Feature                                  Importance
----------------------------------------------------
category_idx                                 0.5422
cont_scaled_2                                0.1525
cont_scaled_0                                0.1071
cont_scaled_1                                0.0707
country_idx                                  0.0564
currency_idx                                 0.0276
goal_bucket_ohe_medium                       0.0097
launch_month_ohe_7                           0.0079
launch_day_of_week_ohe_3                     0.0065
launch_month_ohe_12                          0.0046
launch_month_ohe_1                           0.0024
launch_day_of_week_ohe_7                     0.0022
launch_day_of_week_ohe_6                     0.0019
launch_month_ohe_8                           0.0013
main_category_ohe_Theater                    0.0012


## 7.4 Model Comparison

In [32]:
# Regression model comparison
regression_metrics = [lin_metrics, gbt_metrics]

print(f"{'Model':<26} {'RMSE(log)':>10} {'R2(log)':>9} {'MAE(log)':>9} {'RMSE($)':>14} {'MAE($)':>14}")
print("-" * 86)
for m in regression_metrics:
    print(f"{m['model']:<26} {m['rmse_log']:>10.4f} {m['r2_log']:>9.4f} {m['mae_log']:>9.4f} {m['rmse_orig']:>14,.0f} {m['mae_orig']:>14,.0f}")

r2_gain = gbt_metrics['r2_log'] - lin_metrics['r2_log']
print(f"\nGBT vs Linear Regression: R2 (log) change {r2_gain:+.4f}")

Model                       RMSE(log)   R2(log)  MAE(log)        RMSE($)         MAE($)
--------------------------------------------------------------------------------------
Linear Regression              3.1198    0.0863    2.5654        102,722         10,216
GBT Regressor (tuned)          2.8747    0.2242    2.3122        102,142          9,885

GBT vs Linear Regression: R2 (log) change +0.1379


### Phase 7 Checkpoint Summary

- Baseline Linear Regression and tuned GBT trained; evaluated on the held-out test set.
- Reported RMSE / R² / MAE on both log and original-dollar scales.
- Extracted GBT feature importances (`regression_feature_importance`).



---
# Phase 8: Clustering (K-Means)

**Objective:** Segment campaigns into natural groups using the rich (unsupervised) feature profile from Phase 5.4: `goal`, `usd_pledged_real`, `backers`, `campaign_duration_days`, `funding_ratio` (scaled).


## 8.1 Elbow Method

Fit K-Means for k = 2..10 and record the within-cluster sum of squared errors (WCSS / `trainingCost`). The "elbow" - where additional clusters stop meaningfully reducing WCSS - indicates a good k.

In [33]:
# Elbow method -> WCSS for k = 2..10
wcss = []
for k in range(2, 11):
    km = KMeans(featuresCol="features", k=k, seed=42)
    km_model = km.fit(clustering_prepared)
    cost = km_model.summary.trainingCost   # within-set sum of squared errors
    wcss.append((k, cost))
    print(f"k={k:2d}  WCSS={cost:,.1f}")

# % reduction in WCSS as k increases (helps spot the elbow)
print("\nMarginal WCSS reduction:")
for i in range(1, len(wcss)):
    prev, curr = wcss[i-1][1], wcss[i][1]
    print(f"  k={wcss[i][0]:2d}: {(prev-curr)/prev*100:5.1f}% drop vs k={wcss[i-1][0]}")

k= 2  WCSS=1,386,733.3


k= 3  WCSS=1,089,260.8


k= 4  WCSS=1,001,955.2


k= 5  WCSS=637,434.6


k= 6  WCSS=502,621.7


k= 7  WCSS=456,934.0


k= 8  WCSS=415,227.9


k= 9  WCSS=372,071.8


k=10  WCSS=356,019.2

Marginal WCSS reduction:
  k= 3:  21.5% drop vs k=2
  k= 4:   8.0% drop vs k=3
  k= 5:  36.4% drop vs k=4
  k= 6:  21.1% drop vs k=5
  k= 7:   9.1% drop vs k=6
  k= 8:   9.1% drop vs k=7
  k= 9:  10.4% drop vs k=8
  k=10:   4.3% drop vs k=9


## 8.2 K-Means at Optimal k + Silhouette

Based on the elbow, **k = 4** balances compactness against simplicity (WCSS reduction flattens after ~4). Fit the final model, assign every campaign to a cluster, and measure cluster quality with the silhouette score (range -1..1; higher = better separated).

In [34]:
# Final K-Means at optimal k + silhouette
OPTIMAL_K = 4   # chosen from the elbow above

kmeans = KMeans(featuresCol="features", k=OPTIMAL_K, seed=42)
kmeans_model = kmeans.fit(clustering_prepared)
clustered_full = kmeans_model.transform(clustering_prepared)   # has 'features' + 'prediction'

# silhouette needs the 'features' vector
silhouette = ClusteringEvaluator(featuresCol="features", predictionCol="prediction").evaluate(clustered_full)

# Lean cached copy (drop heavy vector cols) for profiles + export -> stable, no recompute
clustered = clustered_full.drop("clust_vec", "features").cache()
clustered.count()

print(f"Optimal k         : {OPTIMAL_K}")
print(f"Silhouette score  : {silhouette:.4f}")
print("\nCluster sizes:")
clustered.groupBy("prediction").count().orderBy("prediction").show()

Optimal k         : 4
Silhouette score  : 0.6473

Cluster sizes:
+----------+------+
|prediction| count|
+----------+------+
|         0|265228|
|         1|   547|
|         2|    26|
|         3| 65874|
+----------+------+



## 8.3 Cluster Profiles

Average characteristics per cluster - including the **success rate** (`avg(label)`) - to interpret what each segment represents (e.g. small under-funded campaigns vs. viral over-funded ones).

In [35]:
# Cluster profiles -> write enriched table via Spark JVM writer, then aggregate in pandas
import os, glob, shutil
from pyspark.sql.functions import year

OUT_DIR = "d:/Softwarica/Big Data/BigDataAssignment/Output"
os.makedirs(OUT_DIR, exist_ok=True)

def spark_write_single_csv(sdf, name):
    """Write one clean CSV via Spark's JVM writer (no driver collect / toPandas)."""
    tmp = os.path.join(OUT_DIR, "_tmp_" + name)
    sdf.coalesce(1).write.option("header", True).option("quote", '"').option("escape", '"').mode("overwrite").csv(tmp)
    part = glob.glob(os.path.join(tmp, "part-*.csv"))[0]
    dest = os.path.join(OUT_DIR, name)
    if os.path.exists(dest):
        os.remove(dest)
    shutil.move(part, dest)
    shutil.rmtree(tmp)
    return dest

# Per-campaign enriched table (~331,675 rows): raw dims + engineered + cluster id
campaigns_enriched_sdf = clustered.select(
    "ID", "category", "main_category", "currency", "country", "state", "label",
    "goal", "pledged", "usd_pledged_real", "usd_goal_real", "backers",
    year(col("launched_ts")).alias("launched_year"),
    "launch_month", "launch_day_of_week", "campaign_duration_days",
    "funding_ratio", "goal_bucket", "is_overfunded",
    col("prediction").alias("cluster"),
)
spark_write_single_csv(campaigns_enriched_sdf, "campaigns_enriched.csv")
print("campaigns_enriched.csv written (Spark JVM writer).")

# Read back (~36 MB) and compute cluster profiles in pandas (no Spark shuffle)
enriched_pd = pd.read_csv(os.path.join(OUT_DIR, "campaigns_enriched.csv"))
print(f"Loaded enriched: {enriched_pd.shape[0]:,} rows x {enriched_pd.shape[1]} cols")

cluster_profiles_pd = (enriched_pd.groupby("cluster").agg(
        size=("ID", "count"), avg_goal=("goal", "mean"),
        avg_pledged=("usd_pledged_real", "mean"), avg_backers=("backers", "mean"),
        avg_duration_days=("campaign_duration_days", "mean"),
        avg_funding_ratio=("funding_ratio", "mean"), success_rate=("label", "mean"))
    .round(3).reset_index())
print("\n=== Cluster Profiles ===")
print(cluster_profiles_pd.to_string(index=False))

# Cluster centers (standardized space) straight from the model -> pandas
cluster_centers_pd = pd.DataFrame(
    [[i] + [float(v) for v in c] for i, c in enumerate(kmeans_model.clusterCenters())],
    columns=["cluster"] + clust_features,
)
print("\n=== Cluster Centers (scaled space) ===")
print(cluster_centers_pd.round(3).to_string(index=False))

campaigns_enriched.csv written (Spark JVM writer).


Loaded enriched: 331,675 rows x 20 cols

=== Cluster Profiles ===
 cluster   size   avg_goal  avg_pledged  avg_backers  avg_duration_days  avg_funding_ratio  success_rate
       0 265228  23524.519     7554.158       97.250             28.692              2.672         0.427
       1    547 156890.810  1167746.700    10604.861             35.927             25.662         0.996
       2     26 590384.615  6936819.474    73712.962             36.462            127.765         1.000
       3  65874 126553.879     7215.418       77.269             55.128              6.651         0.305

=== Cluster Centers (scaled space) ===
 cluster   goal  usd_pledged_real  backers  campaign_duration_days  funding_ratio
       0 -0.019            -0.025   -0.020                  -0.414         -0.003
       1  0.101            12.009   10.886                   0.156          0.079
       2  0.489            71.608   76.232                   0.197          0.440
       3  0.074            -0.028   -0.04

### Phase 8 Checkpoint Summary

- Elbow method computed WCSS for k = 2..10; chose **k = 4**.
- Fit final K-Means (seed=42); reported silhouette score.
- Built `cluster_profiles` and `cluster_centers`; `clustered` holds per-campaign assignments.


---
# Phase 9: Results Export for Tableau

Write all model results as clean single-file CSVs into `Output/` for visualisation in Tableau. Each Spark DataFrame is converted to pandas only at this final export step (the sanctioned use of pandas - all heavy processing stayed in Spark).

**Files produced (9):** classification results / metrics / feature importance, regression results / metrics / feature importance, clustering assignments / profiles / centers.

In [36]:
# Results export for Tableau -> small data via pandas, large via Spark JVM writer
from pyspark.ml.functions import vector_to_array
print(f"Exporting all result CSVs to: {OUT_DIR}")

Exporting all result CSVs to: d:/Softwarica/Big Data/BigDataAssignment/Output


In [37]:
# Export classification
print("Classification:")
# per-test-campaign predictions + probability of success (Spark JVM write, no toPandas)
clf_results = (rf_pred
    .select("ID", "label", "prediction", vector_to_array("probability").alias("_p"))
    .withColumn("prob_success", col("_p")[1]).drop("_p"))
spark_write_single_csv(clf_results, "classification_results.csv")
# metrics + feature importance straight from Python objects (no Spark)
pd.DataFrame(classification_metrics).to_csv(os.path.join(OUT_DIR, "classification_metrics.csv"), index=False)
pd.DataFrame(fi_pairs, columns=["feature", "importance"]).to_csv(
    os.path.join(OUT_DIR, "classification_feature_importance.csv"), index=False)
print("  classification_results.csv, classification_metrics.csv, classification_feature_importance.csv")

Classification:


  classification_results.csv, classification_metrics.csv, classification_feature_importance.csv


In [38]:
# Export regression
print("Regression:")
reg_results = (gbt_pred
    .select("ID", "pledged", col("prediction").alias("pledged_log_pred"))
    .withColumn("pledged_pred", expm1(col("pledged_log_pred"))))
spark_write_single_csv(reg_results, "regression_results.csv")
pd.DataFrame(regression_metrics).to_csv(os.path.join(OUT_DIR, "regression_metrics.csv"), index=False)
pd.DataFrame(gbt_fi_pairs, columns=["feature", "importance"]).to_csv(
    os.path.join(OUT_DIR, "regression_feature_importance.csv"), index=False)
print("  regression_results.csv, regression_metrics.csv, regression_feature_importance.csv")

Regression:


  regression_results.csv, regression_metrics.csv, regression_feature_importance.csv


In [39]:
# Export clustering + list all files
print("Clustering:")
# clustering_assignments: subset of the enriched table (already in pandas)
enriched_pd[["ID", "cluster", "goal", "usd_pledged_real", "backers",
             "campaign_duration_days", "funding_ratio"]].to_csv(
    os.path.join(OUT_DIR, "clustering_assignments.csv"), index=False)
cluster_profiles_pd.to_csv(os.path.join(OUT_DIR, "cluster_profiles.csv"), index=False)
cluster_centers_pd.to_csv(os.path.join(OUT_DIR, "cluster_centers.csv"), index=False)
# elbow (k, wcss) from the Phase 8.1 list
pd.DataFrame(wcss, columns=["k", "wcss"]).to_csv(os.path.join(OUT_DIR, "clustering_elbow.csv"), index=False)
print("  clustering_assignments.csv, cluster_profiles.csv, cluster_centers.csv, clustering_elbow.csv")

print("\nAll files in Output/:")
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    if os.path.isfile(p):
        print(f"  {f:<42} {os.path.getsize(p):>10,} bytes")

Clustering:


  clustering_assignments.csv, cluster_profiles.csv, cluster_centers.csv, clustering_elbow.csv

All files in Output/:
  campaigns_enriched.csv                     36,918,181 bytes
  classification_feature_importance.csv           2,068 bytes
  classification_metrics.csv                        331 bytes
  classification_results.csv                  2,434,681 bytes
  cluster_centers.csv                               488 bytes
  cluster_profiles.csv                              320 bytes
  clustering_assignments.csv                 14,505,705 bytes
  clustering_elbow.csv                              203 bytes
  regression_feature_importance.csv               1,915 bytes
  regression_metrics.csv                            281 bytes
  regression_results.csv                      3,596,451 bytes


### Phase 9 Checkpoint Summary

- Exported 9 CSV files to `Output/` for Tableau.
- Classification: per-campaign predictions + success probability, metrics, feature importance.
- Regression: per-campaign predicted pledge (log + dollars), metrics, feature importance.
- Clustering: per-campaign cluster assignments, cluster profiles, cluster centers.


---
# Phase 10: Reproducibility Verification & Execution Summary

Final consolidation: confirm the reproducibility guarantees and print a single summary of the whole pipeline and all results.

In [40]:
# Execution summary
print("=" * 64)
print(" KICKSTARTER CAMPAIGN ANALYTICS - EXECUTION SUMMARY")
print("=" * 64)
print(f" Random seed (everywhere)      : {SEED}")
print(f" Raw rows loaded               : {n_rows:,}")
print(f" Rows after cleaning           : {df_model.count():,}")
print(f" Train / Test split (80/20)    : {n_train:,} / {n_test:,}")
print(f" Expanded feature vector size  : {len(feature_names)}")
print("-" * 64)
print(" Reproducibility guarantees:")
print("   - seed=42 on split, all models, and cross-validation")
print("   - preprocessing pipeline fit on TRAIN only (no scaler leakage)")
print("   - supervised features exclude pledged/backers-derived columns")
print("-" * 64)
print(" CLASSIFICATION (test set):")
for m in classification_metrics:
    print(f"   {m['model']:<26} acc={m['accuracy']:.4f}  f1={m['f1']:.4f}  auc={m['auc']:.4f}")
print(" REGRESSION (test set):")
for m in regression_metrics:
    print(f"   {m['model']:<26} R2(log)={m['r2_log']:.4f}  RMSE($)={m['rmse_orig']:,.0f}")
print(f" CLUSTERING: k={OPTIMAL_K}, silhouette={silhouette:.4f}")
print("=" * 64)
print(" All result CSVs exported to Output/ for Tableau.")
print(" Pipeline complete and reproducible.")
print("=" * 64)

 KICKSTARTER CAMPAIGN ANALYTICS - EXECUTION SUMMARY
 Random seed (everywhere)      : 42
 Raw rows loaded               : 378,661
 Rows after cleaning           : 331,675
 Train / Test split (80/20)    : 265,265 / 66,410
 Expanded feature vector size  : 49
----------------------------------------------------------------
 Reproducibility guarantees:
   - seed=42 on split, all models, and cross-validation
   - preprocessing pipeline fit on TRAIN only (no scaler leakage)
   - supervised features exclude pledged/backers-derived columns
----------------------------------------------------------------
 CLASSIFICATION (test set):
   Logistic Regression        acc=0.7197  f1=0.7091  auc=0.7939
   Random Forest (tuned)      acc=0.8510  f1=0.8521  auc=0.9222
 REGRESSION (test set):
   Linear Regression          R2(log)=0.0863  RMSE($)=102,722
   GBT Regressor (tuned)      R2(log)=0.2242  RMSE($)=102,142
 CLUSTERING: k=4, silhouette=0.6473
 All result CSVs exported to Output/ for Tableau.
 Pipelin

### Phase 10 Checkpoint Summary

All 10 phases implemented:
1. Data Loading & Validation
2. Data Cleaning & Preprocessing
3. Feature Engineering (6 features)
4. Feature Preprocessing (leakage-safe pipeline)
5. Train-Test Split & Pipeline Fitting
6. Classification (Logistic Regression + Random Forest)
7. Regression (Linear Regression + GBT)
8. Clustering (K-Means)
9. Results Export (9 CSVs for Tableau)
